# 1. Feature Extraction & Dataset Preparation
This notebook downloads the **MSR-VTT** dataset from Hugging Face, loads the pre-trained multimodal encoders, extracts visual, audio, and video teacher features, and caches them for model training.


## Step 1: Install remote dependencies
We install the required libraries directly on the Kaggle GPU instance.


In [1]:
!pip install -q git+https://github.com/facebookresearch/ImageBind.git
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q decord pytorchvideo resampy soundfile tqdm


  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 81.8 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 54.5 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires num

## Step 2: Import libraries & Verify GPU


In [2]:
import os
import zipfile
import urllib.request
import json
import torch
import numpy as np
import PIL.Image as Image
from tqdm import tqdm
import torchaudio
import decord
from decord import VideoReader, cpu

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"Decord GPU support: {decord.__file__}")


Using device: cuda
Decord GPU support: /usr/local/lib/python3.12/dist-packages/decord/__init__.py


## Step 3: Download and Extract MSR-VTT (Hugging Face Mirror)
We download a standard 1K test split and 7K training split.


In [3]:
os.makedirs("msrvtt", exist_ok=True)

# Download files
urls = {
    "msrvtt/msrvtt_train_7k.json": "https://huggingface.co/datasets/friedrichor/MSR-VTT/resolve/main/msrvtt_train_7k.json",
    "msrvtt/msrvtt_test_1k.json": "https://huggingface.co/datasets/friedrichor/MSR-VTT/resolve/main/msrvtt_test_1k.json",
    "msrvtt/MSRVTT_Videos.zip": "https://huggingface.co/datasets/friedrichor/MSR-VTT/resolve/main/MSRVTT_Videos.zip"
}

for path, url in urls.items():
    if not os.path.exists(path):
        print(f"Downloading {path}...")
        urllib.request.urlretrieve(url, path)
        print("Done.")

# Unzip videos
video_dir = "msrvtt/video"
if not os.path.exists(video_dir):
    print("Extracting videos...")
    with zipfile.ZipFile("msrvtt/MSRVTT_Videos.zip", 'r') as zip_ref:
        zip_ref.extractall("msrvtt")
    print("Done extraction.")


Done.
Done.
Done.
Extracting videos...
Done extraction.


## Step 4: Define Encoders and Preprocessing Helpers
We load CLIP vision (student visual), VGGish (student audio), and ImageBind (teacher video).


In [4]:
import clip
from imagebind.models import imagebind_model
from imagebind.models.imagebind_model import ModalityType
import imagebind.data as ib_data

# 1. CLIP Vision
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()

# 2. VGGish via torchhub
vggish = torch.hub.load('harritaylor/torchvggish', 'vggish', trust_repo=True)
vggish.eval()
vggish.to(device)

# 3. ImageBind Video (Teacher)
ib_model = imagebind_model.imagebind_huge(pretrained=True)
ib_model.eval()
ib_model.to(device)

print("Encoders loaded successfully!")


100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 132MiB/s]


Downloading: "https://github.com/harritaylor/torchvggish/zipball/master" to /root/.cache/torch/hub/master.zip
Downloading: "https://github.com/harritaylor/torchvggish/releases/download/v0.1/vggish-10086976.pth" to /root/.cache/torch/hub/checkpoints/vggish-10086976.pth


100%|██████████| 275M/275M [00:01<00:00, 162MB/s]  


Downloading: "https://github.com/harritaylor/torchvggish/releases/download/v0.1/vggish_pca_params-970ea276.pth" to /root/.cache/torch/hub/checkpoints/vggish_pca_params-970ea276.pth


100%|██████████| 177k/177k [00:00<00:00, 7.98MB/s]


100%|██████████| 4.47G/4.47G [00:22<00:00, 209MB/s] 


Encoders loaded successfully!


## Step 5: Define Extraction Helper Functions
We write robust helpers to extract visual frame, audio waveform, and teacher video embeddings.


In [5]:
def extract_clip_frame(video_path):
    # Reads the middle frame and passes it through CLIP Vision
    vr = VideoReader(video_path, ctx=cpu(0))
    mid_idx = len(vr) // 2
    frame = vr[mid_idx]
    if hasattr(frame, 'asnumpy'):
        frame = frame.asnumpy()
    else:
        frame = frame.cpu().numpy()
    pil_img = Image.fromarray(frame)
    img_tensor = clip_preprocess(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = clip_model.encode_image(img_tensor)
    return feat.squeeze(0).cpu()

def extract_vggish_audio(video_path):
    # Extracts audio from video path and computes 128-dim VGGish embedding
    import subprocess
    temp_wav = "temp_audio.wav"
    if os.path.exists(temp_wav):
        os.remove(temp_wav)
    # Resample to 16kHz mono using ffmpeg
    cmd = f"ffmpeg -y -i {video_path} -vn -acodec pcm_s16le -ar 16000 -ac 1 {temp_wav}"
    subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    # If no audio track, return zero vector
    if not os.path.exists(temp_wav) or os.path.getsize(temp_wav) < 1000:
        if os.path.exists(temp_wav): os.remove(temp_wav)
        return torch.zeros(128)
        
    try:
        with torch.no_grad():
            feat = vggish.forward(temp_wav)
            if feat.ndim > 1:
                feat = feat.mean(dim=0) # Aggregate across frames
        os.remove(temp_wav)
        return feat.cpu()
    except Exception as e:
        if os.path.exists(temp_wav): os.remove(temp_wav)
        return torch.zeros(128)

def extract_imagebind_video(video_path):
    # Passes the full video through ImageBind video pipeline
    inputs = {
        ModalityType.VISION: ib_data.load_and_transform_video_data([video_path], device)
    }
    with torch.no_grad():
        embeddings = ib_model(inputs)
        feat = embeddings[ModalityType.VISION]
    return feat.squeeze(0).cpu()


## Step 6: Feature Extraction Loop
We extract features for both training and testing datasets. Since this takes time, we save progress incrementally.


In [6]:
def run_extraction(split_json, output_pt):
    with open(split_json) as f:
        data = json.load(f)
    
    # Limit to unique videos (MSR-VTT contains multiple sentences per video)
    unique_videos = {}
    for item in data:
        unique_videos[item['video_id']] = item['video']
        
    video_ids = sorted(list(unique_videos.keys()))
    print(f"Extracting features for {len(video_ids)} videos from {split_json}...")
    
    z_imgs = []
    z_auds = []
    v_teachers = []
    valid_video_ids = []
    
    for vid in tqdm(video_ids):
        video_file = os.path.join("msrvtt/video", unique_videos[vid])
        if not os.path.exists(video_file):
            continue
        
        try:
            # Extract visual
            z_img = extract_clip_frame(video_file)
            # Extract audio
            z_aud = extract_vggish_audio(video_file)
            # Extract teacher video
            v_teacher = extract_imagebind_video(video_file)
            
            z_imgs.append(z_img)
            z_auds.append(z_aud)
            v_teachers.append(v_teacher)
            valid_video_ids.append(vid)
        except Exception as e:
            print(f"Error processing {vid}: {e}")
            continue
            
    # Convert to tensors
    out_dict = {
        'z_img': torch.stack(z_imgs),
        'z_aud': torch.stack(z_auds),
        'v_teacher': torch.stack(v_teachers),
        'video_ids': valid_video_ids
    }
    torch.save(out_dict, output_pt)
    print(f"Saved to {output_pt}. Shape z_img: {out_dict['z_img'].shape}, z_aud: {out_dict['z_aud'].shape}, v_teacher: {out_dict['v_teacher'].shape}")


## Step 7: Run extraction
Let's extract features for both training and test sets.


In [19]:
run_extraction("msrvtt/msrvtt_test_1k.json", "test_features.pt")
run_extraction("msrvtt/msrvtt_train_7k.json", "train_features.pt")


Extracting features for 1000 videos from msrvtt/msrvtt_test_1k.json...


100%|██████████| 1000/1000 [34:49<00:00,  2.09s/it]


Saved to test_features.pt. Shape z_img: torch.Size([1000, 512]), z_aud: torch.Size([1000, 128]), v_teacher: torch.Size([1000, 1024])
Extracting features for 7010 videos from msrvtt/msrvtt_train_7k.json...


 95%|█████████▌| 6672/7010 [3:55:11<11:51,  2.11s/it]  

: 

In [7]:
import os
import shutil
from IPython.display import HTML, display, FileLink

# 1. Run the feature extraction
run_extraction("msrvtt/msrvtt_train_7k.json", "train_features.pt")

# 2. Trigger download flow immediately upon completion
file_name = "train_features.pt"

if os.path.exists(file_name):
    file_size_mb = os.path.getsize(file_name) / (1024 * 1024)
    print(f"\n[SUCCESS] Feature extraction completed! Saved: {file_name} ({file_size_mb:.2f} MB)")
    
    # Mechanism 1: Display a fallback manual download link
    display(FileLink(file_name, result_html_prefix="Manual download link: "))
    
    # Mechanism 2: Auto-trigger browser download via JavaScript (For Kaggle / Jupyter Web UI)
    js_code = f"""
    <div style="padding: 12px; background-color: #d4edda; color: #155724; border: 1px solid #c3e6cb; border-radius: 6px; margin: 15px 0;">
        <strong>🎉 Extraction Complete!</strong> Auto-downloading <code>{file_name}</code> to your local machine...
        <br><small>If the download didn't start automatically, click the manual download link above or check your browser's pop-up blocker.</small>
    </div>
    <script>
    (function() {{
        var path = window.location.pathname;
        var baseIdx = path.indexOf('/notebooks/');
        if (baseIdx === -1) baseIdx = path.indexOf('/lab');
        if (baseIdx === -1) baseIdx = path.indexOf('/tree');
        
        var basePath = baseIdx !== -1 ? path.substring(0, baseIdx + 1) : '/';
        var downloadUrl = window.location.origin + basePath + 'files/{file_name}';
        
        console.log("Auto-downloading from: " + downloadUrl);
        
        var a = document.createElement('a');
        a.href = downloadUrl;
        a.download = '{file_name}';
        document.body.appendChild(a);
        a.click();
        document.body.removeChild(a);
    }})();
    </script>
    """
    display(HTML(js_code))
    
    # Mechanism 3: Local CLI sync command template
    print("\n" + "="*80)
    print("ALTERNATIVE: Sync features locally using the project sync script:")
    print("Run this command in your local terminal (replace <URL> with your Kaggle connection URL):")
    print(f"python src/sync_workspace.py --action download_features --url \"<YOUR_KAGGLE_URL>\"")
    print("="*80 + "\n")
else:
    print(f"\n[ERROR] File {file_name} was not found in the working directory.")

Extracting features for 7010 videos from msrvtt/msrvtt_train_7k.json...


100%|██████████| 7010/7010 [3:50:09<00:00,  1.97s/it]  

Saved to train_features.pt. Shape z_img: torch.Size([7010, 512]), z_aud: torch.Size([7010, 128]), v_teacher: torch.Size([7010, 1024])

[SUCCESS] Feature extraction completed! Saved: train_features.pt (37.78 MB)


/kaggle/working/train_features.pt


ALTERNATIVE: Sync features locally using the project sync script:
Run this command in your local terminal (replace <URL> with your Kaggle connection URL):
python src/sync_workspace.py --action download_features --url "<YOUR_KAGGLE_URL>"



In [12]:
import os
print("Current Working Directory:", os.getcwd())
print("Files in Current Directory:", os.listdir('.'))
if os.path.exists('msrvtt'):
    print("Files in 'msrvtt':", os.listdir('msrvtt'))
    for folder in ['video', 'MSRVTT_Videos']:
        path = os.path.join('msrvtt', folder)
        if os.path.exists(path):
            files = os.listdir(path)
            print(f"Folder '{path}' exists and contains {len(files)} files.")
            if files:
                print(f"Sample files in '{path}':", files[:5])
        else:
            print(f"Folder '{path}' does NOT exist.")
else:
    print("'msrvtt' folder does NOT exist.")

Current Working Directory: /kaggle/working
Files in Current Directory: ['msrvtt', '.virtual_documents', '.checkpoints']
Files in 'msrvtt': ['MSRVTT_Videos.zip', 'msrvtt_test_1k.json', 'msrvtt_train_7k.json', 'video']
Folder 'msrvtt/video' exists and contains 10000 files.
Sample files in 'msrvtt/video': ['video1106.mp4', 'video7941.mp4', 'video4587.mp4', 'video8880.mp4', 'video7930.mp4']
Folder 'msrvtt/MSRVTT_Videos' does NOT exist.


In [8]:
import os
import sys
import torch
from IPython.display import FileLink, display

# 1. Check if the file was already successfully saved in the background
if os.path.exists("train_features.pt"):
    print("🎉 Success! 'train_features.pt' already exists on the remote disk.")
    size_mb = os.path.getsize("train_features.pt") / (1024 * 1024)
    print(f"File size: {size_mb:.2f} MB")
    try:
        data = torch.load("train_features.pt", map_location="cpu")
        print("Successfully loaded file structure:")
        for k, v in data.items():
            if isinstance(v, torch.Tensor):
                print(f"  - {k}: shape {v.shape}")
            else:
                print(f"  - {k}: length {len(v)}")
    except Exception as e:
        print(f"Error loading train_features.pt: {e}")
else:
    print("File not found on disk. Attempting to recover accumulated features from the Jupyter traceback variables...")
    
    # 2. Recover from the interrupted function frame traceback
    tb = getattr(sys, 'last_traceback', None)
    if tb is None:
        print("❌ Error: No traceback found in 'sys.last_traceback'.")
        print("Note: If you ran another failing cell since the disconnection, the traceback might have been cleared.")
    else:
        found = False
        while tb is not None:
            frame = tb.tb_frame
            if frame.f_code.co_name == 'run_extraction':
                print("🔍 Found the active 'run_extraction' frame in memory!")
                locals_dict = frame.f_locals
                
                # Retrieve the lists accumulated before the disconnect
                z_imgs = locals_dict.get('z_imgs', [])
                z_auds = locals_dict.get('z_auds', [])
                v_teachers = locals_dict.get('v_teachers', [])
                valid_video_ids = locals_dict.get('valid_video_ids', [])
                
                print(f"Recovered features count:")
                print(f"  - z_imgs: {len(z_imgs)}")
                print(f"  - z_auds: {len(z_auds)}")
                print(f"  - v_teachers: {len(v_teachers)}")
                print(f"  - valid_video_ids: {len(valid_video_ids)}")
                
                min_len = min(len(z_imgs), len(z_auds), len(v_teachers), len(valid_video_ids))
                if min_len > 0:
                    print(f"Stacking and saving the first {min_len} recovered video features...")
                    out_dict = {
                        'z_img': torch.stack(z_imgs[:min_len]),
                        'z_aud': torch.stack(z_auds[:min_len]),
                        'v_teacher': torch.stack(v_teachers[:min_len]),
                        'video_ids': valid_video_ids[:min_len]
                    }
                    
                    torch.save(out_dict, "train_features.pt")
                    print("🎉 Successfully recovered and saved features to 'train_features.pt'!")
                else:
                    print("❌ Error: Recovered feature lists are empty.")
                
                found = True
                break
            tb = tb.tb_next
            
        if not found:
            print("❌ 'run_extraction' function frame was not found in the active traceback.")

# 3. Provide direct download link
if os.path.exists("train_features.pt"):
    print("\n⬇️ Click the link below to download the file directly through your browser:")
    display(FileLink("train_features.pt"))
else:
    print("\nCould not recover or find 'train_features.pt'.")

🎉 Success! 'train_features.pt' already exists on the remote disk.
File size: 37.78 MB
Successfully loaded file structure:
  - z_img: shape torch.Size([7010, 512])
  - z_aud: shape torch.Size([7010, 128])
  - v_teacher: shape torch.Size([7010, 1024])
  - video_ids: length 7010

⬇️ Click the link below to download the file directly through your browser:


/kaggle/working/train_features.pt